In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
MLR_OUTPUT_CSV = "P4_Nested_MLR_Metrics.csv"
SVR_OUTPUT_CSV = "P4_Nested_SVR_Metrics.csv"
XGB_OUTPUT_CSV = "P4_Nested_XGBoost_Metrics.csv"
COMPARISON_CSV = "P4_Model_Comparison_Summary.csv"

# ==========================================
# 分别控制每个模型 —— 设 True 才执行，互不依赖
# 全部跑完后将 RUN_SUMMARY 改为 True 生成对比表
# ==========================================
RUN_MLR = False
RUN_SVR = False
RUN_XGB = True
RUN_SUMMARY = True

# Optuna 迭代次数
OPTUNA_TRIALS_MLR = 30   # MLR 无超参，仅特征选择
OPTUNA_TRIALS_SVR = 20   # [加速] SVR 对大数据集极慢，降至 20
OPTUNA_TRIALS_XGB = 50

SVR_MAX_SAMPLES = 2000    # [加速] SVR Optuna 搜索阶段最大样本数 (O(n^2))
RANDOM_SEED = 42
# ==========================================

In [2]:
def calculate_metrics(y_true, y_pred):
    """与原 RF+MOBO 完全一致的指标计算"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 if mean_true != 0 else 0
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

In [3]:
def run_loyo_mobo(model_name, objective_func, output_csv, n_trials, extra_fixed_params=None):
    """
    通用 LOYO + Optuna MOBO 执行器
    
    参数:
        model_name: 模型名称 (用于打印)
        objective_func: callable(trial, X_df, y_np, groups) -> (mae, n_features)
        output_csv: 输出 CSV 路径
        n_trials: Optuna 迭代次数
        extra_fixed_params: 额外固定参数 dict（例如 XGBoost 的 tree_method）
    """
    if not os.path.exists(INPUT_FILE):
        print(f"错误: 找不到文件 {INPUT_FILE}")
        return
    
    df = pd.read_csv(INPUT_FILE)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*75}")
    print(f">>> 启动 {model_name} MOBO 联合优化")
    print(f"    初始特征维度: {len(feature_cols)}")
    print(f"    单年寻优次数: {n_trials}")
    print(f"{'='*75}")
    
    for test_year in years:
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_df = train_df[feature_cols]
        y_train_np = train_df['yield'].values
        groups_train = train_df['Year'].values
        X_test_df = test_df[feature_cols]
        y_test_np = test_df['yield'].values
        
        print(f"    [Year {test_year}] 正在寻找 Pareto 最优解...", end="", flush=True)
        
        study = optuna.create_study(
            directions=['minimize', 'minimize'],
            sampler=TPESampler(seed=RANDOM_SEED)
        )
        func = lambda trial: objective_func(trial, X_train_df, y_train_np, groups_train)
        study.optimize(func, n_trials=n_trials)
        
        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]
        
        # 解析特征掩码
        selected_features = [col for col in feature_cols if best_trial.params.get(f'mask_{col}', False)]
        
        print(f" 完成.")
        print(f"      -> 内部评级 MAE: {best_trial.values[0]:.2f} | 选用特征数: {len(selected_features)}/{len(feature_cols)}")
        
        X_train_np_final = X_train_df[selected_features].values
        X_test_np_final = X_test_df[selected_features].values
        y_pred = objective_func.fit_best_model(best_trial, X_train_np_final, y_train_np, X_test_np_final)
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Used_Features'] = len(selected_features)
        m['Best_Params'] = objective_func.format_params(best_trial)
        fold_results.append(m)
        
        print(f"      -> [验证结果] RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")
    
    # 全局指标
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Used_Features'] = 'N/A'
    global_metrics['Best_Params'] = 'N/A'
    fold_results.append(global_metrics)
    
    print(f"{'='*75}")
    print(f">>> {model_name} 全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*75}")
    
    cols = ['Test_Year', 'Used_Features', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    pd.DataFrame(fold_results)[cols].to_csv(output_csv, index=False)
    print(f"指标已保存至: {output_csv}")

## 1. MLR (Multiple Linear Regression)

In [4]:
# ==========================================
# MLR: 仅特征选择，无超参
# ==========================================
class MLRObjective:
    def __init__(self):
        self._best_params = None
        self._selected_features = None
    
    def __call__(self, trial, X_df, y_np, groups):
        active_features = []
        for col in X_df.columns:
            if trial.suggest_categorical(f'mask_{col}', [True, False]):
                active_features.append(col)
        
        if len(active_features) == 0:
            return float('inf'), len(X_df.columns)
        
        logo = LeaveOneGroupOut()
        mae_scores = []
        X_np_subset = X_df[active_features].values
        
        for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
            X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
            y_tr, y_val = y_np[train_idx], y_np[val_idx]
            model = LinearRegression()
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            mae_scores.append(mean_absolute_error(y_val, preds))
        
        return np.mean(mae_scores), len(active_features)
    
    def fit_best_model(self, trial, X_train, y_train, X_test):
        model = LinearRegression()
        model.fit(X_train, y_train)
        return model.predict(X_test)
    
    def format_params(self, trial):
        return 'N/A (linear model)'

# 执行 MLR
if RUN_MLR:
    mlr_obj = MLRObjective()
    run_loyo_mobo('MLR', mlr_obj, MLR_OUTPUT_CSV, OPTUNA_TRIALS_MLR)
else:
    print("\n>>> MLR 已跳过 (RUN_MLR=False)")


>>> MLR 已跳过 (RUN_MLR=False)


## 2. SVR (Support Vector Regression)

In [5]:
# ==========================================
# SVR: 特征选择 + 超参优化 (C, gamma, epsilon 固定)
# [加速策略]
#   1. Optuna 搜索阶段最多采样 SVR_MAX_SAMPLES 条 (SVR O(n^2))
#   2. gamma 范围缩窄避开极小区间，epsilon 固定
#   3. cache_size 增大加速核矩阵计算
# ==========================================
class SVRObjective:
    def __call__(self, trial, X_df, y_np, groups):
        # [加速] 大样本时随机下采样 (仅 Optuna 搜索阶段)
        if len(y_np) > SVR_MAX_SAMPLES:
            rng = np.random.RandomState(RANDOM_SEED)
            sample_idx = rng.choice(len(y_np), SVR_MAX_SAMPLES, replace=False)
            X_df = X_df.iloc[sample_idx]
            y_np = y_np[sample_idx]
            groups = groups[sample_idx]
        
        active_features = []
        for col in X_df.columns:
            if trial.suggest_categorical(f'mask_{col}', [True, False]):
                active_features.append(col)
        
        if len(active_features) == 0:
            return float('inf'), len(X_df.columns)
        
        C = trial.suggest_float('C', 0.1, 10.0, log=True)
        gamma = trial.suggest_float('gamma', 1e-3, 1e-1, log=True)
        epsilon = 0.1  # 固定
        
        logo = LeaveOneGroupOut()
        mae_scores = []
        X_np_subset = X_df[active_features].values
        
        for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
            X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
            y_tr, y_val = y_np[train_idx], y_np[val_idx]
            
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)
            
            model = SVR(C=C, gamma=gamma, epsilon=epsilon, kernel='rbf', cache_size=2000)
            model.fit(X_tr_scaled, y_tr)
            preds = model.predict(X_val_scaled)
            mae_scores.append(mean_absolute_error(y_val, preds))
        
        return np.mean(mae_scores), len(active_features)
    
    def fit_best_model(self, trial, X_train, y_train, X_test):
        C = trial.params.get('C', 1.0)
        gamma = trial.params.get('gamma', 'scale')
        epsilon = 0.1
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        model = SVR(C=C, gamma=gamma, epsilon=epsilon, kernel='rbf', cache_size=2000)
        model.fit(X_train_scaled, y_train)
        return model.predict(X_test_scaled)
    
    def format_params(self, trial):
        return str({
            'C': round(trial.params.get('C', 1.0), 4),
            'gamma': round(trial.params.get('gamma', 0.0), 6),
            'epsilon': 0.1
        })

# 执行 SVR
if RUN_SVR:
    svr_obj = SVRObjective()
    run_loyo_mobo('SVR', svr_obj, SVR_OUTPUT_CSV, OPTUNA_TRIALS_SVR)
else:
    print("\n>>> SVR 已跳过 (RUN_SVR=False)")


>>> SVR 已跳过 (RUN_SVR=False)


## 3. XGBoost (标准回归树)

In [6]:
# ==========================================
# XGBoost: 特征选择 + 超参优化 (非 RF 版本)
# ==========================================
class XGBObjective:
    def __call__(self, trial, X_df, y_np, groups):
        active_features = []
        for col in X_df.columns:
            if trial.suggest_categorical(f'mask_{col}', [True, False]):
                active_features.append(col)
        
        if len(active_features) == 0:
            return float('inf'), len(X_df.columns)
        
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.9),
            'tree_method': 'gpu_hist',
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
            'verbosity': 0
        }
        
        logo = LeaveOneGroupOut()
        mae_scores = []
        X_np_subset = X_df[active_features].values
        
        for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
            X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
            y_tr, y_val = y_np[train_idx], y_np[val_idx]
            
            model = XGBRegressor(**params)
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            mae_scores.append(mean_absolute_error(y_val, preds))
        
        return np.mean(mae_scores), len(active_features)
    
    def fit_best_model(self, trial, X_train, y_train, X_test):
        params = {
            'n_estimators': trial.params['n_estimators'],
            'max_depth': trial.params['max_depth'],
            'learning_rate': trial.params['learning_rate'],
            'subsample': trial.params['subsample'],
            'colsample_bytree': trial.params['colsample_bytree'],
            'tree_method': 'gpu_hist',
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
            'verbosity': 0
        }
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)
        return model.predict(X_test)
    
    def format_params(self, trial):
        return str({
            'max_depth': trial.params['max_depth'],
            'learning_rate': round(trial.params['learning_rate'], 4),
            'subsample': round(trial.params['subsample'], 4),
            'colsample_bytree': round(trial.params['colsample_bytree'], 4),
            'n_estimators': trial.params['n_estimators']
        })

# 执行 XGBoost
if RUN_XGB:
    xgb_obj = XGBObjective()
    run_loyo_mobo('XGBoost', xgb_obj, XGB_OUTPUT_CSV, OPTUNA_TRIALS_XGB)
else:
    print("\n>>> XGBoost 已跳过 (RUN_XGB=False)")


>>> 启动 XGBoost MOBO 联合优化
    初始特征维度: 100
    单年寻优次数: 50
    [Year 2016] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 531.71 | 选用特征数: 53/100
      -> [验证结果] RRMSE: 13.88% | MAPE: 9.32%

    [Year 2017] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 537.33 | 选用特征数: 43/100
      -> [验证结果] RRMSE: 14.66% | MAPE: 12.47%

    [Year 2018] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 549.22 | 选用特征数: 34/100
      -> [验证结果] RRMSE: 12.03% | MAPE: 9.50%

    [Year 2019] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 570.67 | 选用特征数: 43/100
      -> [验证结果] RRMSE: 12.41% | MAPE: 9.53%

    [Year 2020] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 537.38 | 选用特征数: 42/100
      -> [验证结果] RRMSE: 10.31% | MAPE: 8.38%

    [Year 2021] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 564.81 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 10.70% | MAPE: 8.58%

>>> XGBoost 全局汇总
    R2: 0.314 | RMSE: 768.40 | RRMSE: 12.48%
    MAE: 551.91 | MAPE: 9.63% | d-index: 0.714
指标已保存至: P4_Nested_XGBoost_Metrics.csv


## 4. 汇总对比表

In [7]:
if RUN_SUMMARY:
    # ==========================================
    # 汇总所有模型结果
    # ==========================================
    def load_model_results(csv_path, model_name):
        if not os.path.exists(csv_path):
            print(f"警告: 找不到 {csv_path}，跳过 {model_name}")
            return None
        df = pd.read_csv(csv_path)
        df['Model'] = model_name
        return df

    existing_rf_csv = "P4_Nested_MOBO_Metrics_Fast.csv"  # 已有的 RF+MOBO 结果

    models_to_load = [
        (existing_rf_csv, 'RF+MOBO'),
        (MLR_OUTPUT_CSV, 'MLR'),
        (SVR_OUTPUT_CSV, 'SVR'),
        (XGB_OUTPUT_CSV, 'XGBoost'),
    ]

    all_dfs = []
    for path, name in models_to_load:
        df = load_model_results(path, name)
        if df is not None:
            all_dfs.append(df)

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)
        
        year_order = ['2016', '2017', '2018', '2019', '2020', '2021', 'Overall_Pooled']
        model_order = ['RF+MOBO', 'MLR', 'SVR', 'XGBoost']
        combined['Test_Year'] = pd.Categorical(combined['Test_Year'], categories=year_order, ordered=True)
        combined['Model'] = pd.Categorical(combined['Model'], categories=model_order, ordered=True)
        combined = combined.sort_values(['Test_Year', 'Model']).reset_index(drop=True)
        
        metric_cols = ['R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE']
        
        print(f"{'='*100}")
        print("=" * 35 + " 模型对照总览表 " + "=" * 35)
        print(f"{'='*100}")
        
        for metric in metric_cols:
            pivot = combined.pivot(index='Test_Year', columns='Model', values=metric)
            pivot = pivot[model_order]
            print(f"\n--- {metric} ---")
            print(pivot.to_string())
        
        print(f"\n{'='*100}")
        
        summary_rows = []
        for _, row in combined.iterrows():
            r = {'Test_Year': row['Test_Year']}
            for metric in metric_cols:
                r[f"{row['Model']}_{metric}"] = row[metric]
            summary_rows.append(r)
        
        summary_df = pd.DataFrame(summary_rows)
        summary_df = summary_df.groupby('Test_Year', sort=False).first().reset_index()
        
        col_order = ['Test_Year']
        for model in model_order:
            for metric in metric_cols:
                col_order.append(f"{model}_{metric}")
        existing_cols = [c for c in col_order if c in summary_df.columns]
        summary_df = summary_df[existing_cols]
        
        summary_df.to_csv(COMPARISON_CSV, index=False)
        print(f"\n宽表已保存至: {COMPARISON_CSV}")
    else:
        print("没有可用结果进行汇总。")
else:
    print("\n>>> 汇总已跳过 (RUN_SUMMARY=False)")

=================================== 模型对照总览表 ===================================

--- R2 ---
Model           RF+MOBO    MLR    SVR  XGBoost
Test_Year                                     
2016              0.229  0.178  0.220    0.176
2017              0.243  0.294  0.285    0.078
2018              0.461  0.019  0.432    0.418
2019              0.401  0.249  0.353    0.371
2020              0.227 -0.223  0.151    0.357
2021              0.433 -0.079  0.522    0.466
Overall_Pooled    0.345  0.109  0.342    0.314

--- RRMSE(%) ---
Model           RF+MOBO     MLR     SVR  XGBoost
Test_Year                                       
2016             13.428  13.865  13.507   13.885
2017             13.281  12.827  12.909   14.662
2018             11.581  15.625  11.882   12.033
2019             12.122  13.568  12.591   12.414
2020             11.301  14.213  11.847   10.307
2021             11.035  15.218  10.129   10.703
Overall_Pooled   12.189  14.221  12.216   12.480

--- d-index ---
Model    